# OpenAI Evaluation with EvalAI

This notebook shows how to **trace and evaluate OpenAI chat completions** using the EvalAI Python SDK.

You'll learn:
1. Run local assertions on any text (no API key needed)
2. Evaluate a batch of prompts with `openai_chat_eval`
3. Wrap an OpenAI client for automatic tracing

In [ ]:
# Install (run once)
# !pip install "pauly4010-evalai-sdk[openai]"

## 1. Local Assertions (no API key needed)

The `expect()` API lets you validate any LLM output locally — no network, no credentials.

In [ ]:
from evalai_sdk import expect

output = "The capital of France is Paris. It is known for the Eiffel Tower."

# Content checks
print(expect(output).to_contain("Paris"))
print(expect(output).to_contain_keywords(["France", "Paris", "Eiffel"]))
print(expect(output).to_not_contain_pii())

# Quality checks
print(expect(output).to_have_sentiment("neutral"))
print(expect(output).to_be_professional())
print(expect(output).to_have_proper_grammar())

## 2. Batch Evaluation with `openai_chat_eval`

Run a set of test cases against an OpenAI model and get pass/fail + scores.

> Requires `OPENAI_API_KEY` environment variable or pass `api_key=` directly.

In [ ]:
from evalai_sdk import openai_chat_eval, OpenAIChatEvalCase

result = await openai_chat_eval(
    name="geography-knowledge",
    model="gpt-4",
    system_prompt="You are a helpful geography tutor. Keep answers concise.",
    cases=[
        OpenAIChatEvalCase(
            input="What is the capital of France?",
            expected_output="Paris",
            assertions=[
                {"type": "contains_keywords", "value": ["Paris"]},
                {"type": "has_length", "value": {"max": 200}},
            ],
        ),
        OpenAIChatEvalCase(
            input="What is the largest ocean?",
            assertions=[
                {"type": "contains_keywords", "value": ["Pacific"]},
                {"type": "not_contains_pii"},
            ],
        ),
        OpenAIChatEvalCase(
            input="Explain continental drift in one sentence.",
            assertions=[
                {"type": "has_length", "value": {"min": 20, "max": 300}},
                {"type": "sentiment", "value": "neutral"},
            ],
        ),
    ],
)

print(f"\n{'='*50}")
print(f"Suite: {result.name}")
print(f"Score: {result.score:.2f}")
print(f"Passed: {result.passed_count}/{result.total}")
print(f"Duration: {result.duration_ms:.0f}ms")

## 3. Inspect Individual Results

In [ ]:
for r in result.results:
    status = "PASS" if r.passed else "FAIL"
    print(f"[{status}] {r.input[:50]}...")
    print(f"  Output: {r.output[:100]}...")
    print(f"  Score: {r.score:.2f} | Duration: {r.duration_ms:.0f}ms")
    if r.assertions:
        for a in r.assertions:
            print(f"    - {a.get('type')}: {'PASS' if a.get('passed') else 'FAIL'}")
    print()

## 4. Drop-in OpenAI Tracing

Wrap your existing OpenAI client to automatically trace every call to the EvalAI platform.

> Requires `EVALAI_API_KEY` for sending traces to the platform.

In [ ]:
from openai import AsyncOpenAI
from evalai_sdk import AIEvalClient
from evalai_sdk.integrations.openai import trace_openai

openai_client = AsyncOpenAI()  # reads OPENAI_API_KEY
eval_client = AIEvalClient.init()  # reads EVALAI_API_KEY

# One line — every call is now traced
traced = trace_openai(openai_client, eval_client)

response = await traced.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": "Explain quantum computing in 2 sentences."}],
)

print(response.choices[0].message.content)

## 5. Validate the Response

Combine tracing with local assertions for a complete eval loop:

In [ ]:
output = response.choices[0].message.content

checks = [
    expect(output).to_contain_keywords(["quantum", "computing"]),
    expect(output).to_have_length(min=20, max=500),
    expect(output).to_not_contain_pii(),
    expect(output).to_be_professional(),
]

for r in checks:
    status = "PASS" if r.passed else "FAIL"
    print(f"[{status}] {r.assertion_type}")

print(f"\n{sum(r.passed for r in checks)}/{len(checks)} checks passed")